# The agent framework

Five agents, each with one job, wired into a pipeline that goes from an RSS feed to a notification:

| agent | what it does |
| --- | --- |
| `ClassicalAgent` | the baseline TF-IDF + Ridge model, cheap and offline |
| `NeighboursAgent` | retrieval only: the geometric mean of comparable prices |
| `FrontierAgent` | RAG plus a language model |
| `SpecialistAgent` | our own QLoRA fine-tune (needs a GPU, so not run here) |
| `EnsembleAgent` | a linear blend of the above, fitted on validation |
| `ScannerAgent` | reads the wine press and structures every wine quoted with a price |
| `PlanningAgent` | scan, price, rank by gap, notify |

In [ ]:
from pricer import vectors
from pricer.agents import (
    ClassicalAgent,
    EnsembleAgent,
    FrontierAgent,
    NeighboursAgent,
    PlanningAgent,
    ScannerAgent,
    price_all,
    setup_logging,
)
from pricer.evaluator import Report, leaderboard
from pricer.items import Wine

setup_logging()
train, val, test = Wine.load_local()
collection, encoder = vectors.load(), vectors.encoder()
members = [ClassicalAgent(), NeighboursAgent(collection, encoder), FrontierAgent(collection, encoder)]

### Fit the blend

On **validation**, never on train: the classical member was fitted on train and the retrieval members
can find train wines verbatim, so their training-set accuracy is fantasy. 150 wines is enough for
five coefficients and keeps the frontier agent's bill small -- and 150 frontier calls is already most
of a free tier's day, so drop it further if you are counting tokens.

In [ ]:
ensemble = EnsembleAgent(members)
ensemble.fit(val[:150])
ensemble.save()
ensemble.price(test[0].description), test[0].price

### Does the blend beat its members?

In [ ]:
sample = test[:100]
guesses = price_all(ensemble, [w.description for w in sample])
scored = sample[: len(guesses)]
if scored:
    Report("Ensemble", [w.label for w in scored], guesses, [w.price for w in scored]).save()
leaderboard()

### The scanner: real wines, in the wild

There is no free live wine-price API, and the deal aggregators carry almost no wine, so the source
here is the wine press: Wine Enthusiast and Decanter RSS. Their articles quote a tasting note and a
shelf price, which is exactly the pair this project needs. See `pricer/deals.py` for what was
verified reachable.

Editorial feeds are noisy: many articles name no price at all, and the scanner throws those away.

In [ ]:
scanner = ScannerAgent()
listings = scanner.scan(per_feed=3)
for listing in listings:
    print(f"${listing.price:>7.0f}  {listing.name}\n          {listing.note[:110]}...")

### The planner, end to end

Scan, drop anything outside the $4-$500 range the models were trained on, price the rest, rank by the
gap, notify on anything big. `memory.json` stops a second run re-reporting the same wine.

A word on the gap: our best model carries an RMSLE near 0.5, so a "$20 bargain" is inside the noise.
The interesting output is the pipeline working, not the trade.

In [ ]:
planner = PlanningAgent(ensemble, scanner=scanner)
opportunities = planner.plan(per_feed=3, threshold=15.0)
for opportunity in opportunities[:10]:
    print(opportunity.summary(), "\n")

### Experiments worth running here

- Add the fine-tuned `SpecialistAgent` to the members (on a GPU box) and refit the blend. Does the
  specialist dominate, or does the ensemble still want the retrieval members?
- Replace the linear blend with gradient boosting over the members' guesses.
- Have the frontier agent output a *range* and use its width as an uncertainty feature for the blend.
- Point the scanner at a retailer's feed instead of the press and see how much of the pipeline still
  works when the prose is marketing copy rather than criticism.
- Run `python app.py` for the Gradio front end, and `scripts/plan.py` for the pipeline on a cron.